#### **Brute Force vs. Gillespie Algorithm** <br> Joshua Pajak, Ph.D. joshua.pajak@umassmed.edu

This Jupyter notebook is intended to introduce the reader to a simple example of a chemical 
reaction simulated with "brute force" and with the Gillespie algorithm. The reader should
come away understanding the concept that we can save considerable computational efforts
if we know how to sample "smarter."

We will work with a toy chemical reaction system $A \xrightarrow{k_1} B \xrightarrow{k_2} C$ that we will model stochastically.
Given some initial concentrations of A, B, and C, can you simulate the population of species over time?

In [ ]:
# --- First, import all libraries we will need ---
import numpy as np              # fast linear algebra 
import random                   # for generating probability distributions
import pandas as pd             # for storing our data in an easy to manipulate manner
import matplotlib.pyplot as plt # for plotting our data

Because this is a simple toy system, we know the deterministic solution, which we will use as a reference to ensure that our stochastic methods are correctly implemented.

In [ ]:
def deterministic_solution(A0, B0, C0, k1, k2, T_MAX, n_points=2000):
    """"
    Implements a deterministic solution of the system A -> B -> C
    Args:
        A0 (int): initial concentration of A
        B0 (int): initial concentration of B
        C0 (int): initial concentration of C
        k1 (float): rate of A -> B
        k2 (float): rate of B -> C
        T_MAX (float): max length of the simulation
        n_points (int, default=2000): granularity of our solution
    Returns:
        pandas DataFrame with ["Time", "A", "B", "C"]
    """
    # --- Initialize ---
    t = np.linspace(0, T_MAX, n_points) # our time array 

    A = A0 * np.exp(-k1 * t) # first order reaction

    if k1 == k2:
        # Special case: identical rates
        B = A0 * k1 * t * np.exp(-k1 * t)
    else:
        # Normal use case: rates are unique
        B = (k1 * A0 / (k2 - k1)) * (np.exp(-k1 * t) - np.exp(-k2 * t)) \
            + B0 * np.exp(-k2 * t)

    C = A0 + B0 + C0 - A - B
    deterministic_df = pd.DataFrame({
        'Time': t,
        'A': A,
        'B': B,
        'C': C
    })
    return deterministic_df

In [ ]:
# --- Sample parameters ---
A0, B0, C0 = 1000, 0, 0
k1, k2 = 0.1, 0.05
T_MAX = 50
# we will use these parameters for all simulations

In [ ]:
# -- Solve the deterministic solution --- 
deterministic_df = deterministic_solution(A0, B0, C0, k1, k2, T_MAX)

In [ ]:
# --- Plot the deterministic solution ---
plt.figure(figsize=(7,5))
plt.plot(deterministic_df["Time"], deterministic_df["A"], 
         color="red", label="A (deterministic)")
plt.plot(deterministic_df["Time"], deterministic_df["B"], 
         color="blue", label="B (deterministic)")
plt.plot(deterministic_df["Time"], deterministic_df["C"], 
         color="black", label="C (deterministic)")
plt.xlabel("time")
plt.ylabel("number of molecules")
plt.legend()
plt.title("Deterministic solution")
plt.show()

**But what if we don't know the analytical solution?** Then we can stochastically simulate our system following simple rules and observe that the correct dynamics fall out! 

One way to do this is to march through time. Initially you ask, "do any reactions occur now?" To do this, we calculate propensity: $a_i = k_i[X_i]$. The probability that _a_ reaction occurs during a fixed amount of time $\Delta t$ timestep is then: $$P_i = a_i\cdot\Delta t$$ The probability that _any_ reaction occurs during a fixed amount of time is the sum of these probabilities $$P_{total} = \sum_{i=1}^{N}P_i = \sum_{i=1}^{N}a_i\cdot\Delta t$$

So we will sample a random number from a uniform distribution: $\text{rand}_1 \in (0, 1]$. If the number is greater than our probability, then _no_ reaction occurs during this fixed time period. If the number is less than our total probability, then _a_ reaction has occured. That is, a reaction occurs if $$\text{rand}_1 < P_{total} $$ and now we just have to figure out which one! So we will roll a second die to determine which reaction. In this case, we only need to compare this random number to the ratios of propensities.

The probability that reaction $i$ is the one that fires is proportional to its individual rate compared to the total rate:$$P(\text{Reaction } i) = \frac{a_i}{a_0}$$ Thus, we select the reaction by drawing a second uniform random number, $\text{rand}_2 \in (0, a_0]$, and finding the reaction index $j$ such that:$$\sum_{i=1}^{j-1} a_i < \text{rand}_2 \le \sum_{i=1}^{j} a_i$$ 

Whether or not a reaction has occured, once that time period is evaluated we can move on to the next time period: we add a fixed value to $\Delta t$ and repeat the process.

There are some caveats of which we need to be aware. 

**Choice of $\Delta t$:** We need to choose a $\Delta t$ small enough such that the probability that _two_ reactions occur within the same $\Delta t$ is vanishingly small (true for Poisson processes). 

**The computational cost:** because $\Delta t$ must be very small, the probability of an event occuring $P_{total}$ is also very small, and this is by design. Hence, over the course of our simulation we calculate a large number of timesteps and in most of these timesteps **nothing happens.** We are wasting a lot of computing power!

In 1977 Daniel Gillespie (based on work in the 1940s by Doob and others), proposed a much more efficient algorithm: what if we can guarantee that we **only simulated the timesteps where something happens?**

In [ ]:
def simulate_brute_force(A0, B0, C0, k1, k2, T_MAX, DT=0.001):
    """
    Simulates the reaction system A -> B -> C using a fixed timestep method.
    Args:
        A0 (int): Initial number of A molecules.
        B0 (int): Initial number of B molecules.
        C0 (int): Initial number of C molecules.
        k1 (float): Rate constant for A -> B.
        k2 (float): Rate constant for B -> C.
        T_MAX (float): Total simulation time.
        DT (float, default = 0.001): Small, fixed time step.
    Returns:
        (pandas.DataFrame, step_count (int), r1_yes (int),  r2_yes (int)) 
        DataFrame ['time', 'A', 'B', 'C']
        step_count is an int keeping track of total number of steps
        r1_yes is total number of steps where reaction 1 (A->B) was successful
        r2_yes is total number of steps where reaction 2 (B->C) was successful
    """

    # --- Initialization ---
    time = 0.0
    step_count = 0 # total number of steps
    r1_yes = 0 # total number of steps where reaction 1 occurs
    r2_yes = 0 # total number of steps where reaction 2 occurs

    # Store the history of species counts and time
    time_history = [time]
    A_history = [A0]
    B_history = [B0]
    C_history = [C0]

    # Current species counts
    A_count = A0
    B_count = B0
    C_count = C0

    # --- Simulation Loop (Brute Force) ---
    while time < T_MAX:
        # --- Calculate Propensities (a_i) ---
        a1 = k1 * A_count # by convention propensities are "a_i"
        a2 = k2 * B_count # if you like you can change to b1 etc.

        # --- Calculate Reaction Probabilities in DT ---
        a_sum = a1 + a2
        if a_sum == 0: # this only occurs when number of A and B are 0
            break
        # --- Throw the dice to see if any reaction has occured ---
        r1 = random.random()
        if r1 < (a_sum * DT) :
            # --- Throw the dice to see which reacion has occured --- 
            r2 = random.random() * a_sum
            if r2 < a1 : # then reaction 1 is chosed
                if A_count > 0:  # we need to ensure there are actually A around to react
                    A_count -= 1 # decrement A by one
                    B_count += 1 # increment B by one
                    r1_yes += 1  # we log this successful step
            else: # otherwise reaction 2 is chosen
                if B_count > 0:  # we still need to ensure B exists
                    B_count -= 1 # decrement B by one
                    C_count += 1 # increment C by one
                    r2_yes += 1  # we log this successful step

        # --- Update Time and History ---
        time += DT
        step_count += 1

        time_history.append(time)
        A_history.append(A_count)
        B_history.append(B_count)
        C_history.append(C_count)

    # --- Convert to Pandas DataFrame ---
    brute_force_df = pd.DataFrame({
        'Time': time_history,
        'A': A_history,
        'B': B_history,
        'C': C_history
    })

    return brute_force_df, step_count, r1_yes, r2_yes

In [ ]:
# --- Run the simulation with same parameters as deterministic ---
brute_force_df, brute_force_steps, successful_r1_steps, successful_r2_steps = simulate_brute_force(A0, B0, C0, k1, k2, T_MAX)

In [ ]:
# --- Plot our species populatiosn over time ---
plt.plot(brute_force_df["Time"], brute_force_df["A"], 
         color="red", label = "A (Brute Force)")
plt.plot(brute_force_df["Time"], brute_force_df["B"], 
         color="blue", label = "B (Brute Force)")
plt.plot(brute_force_df["Time"], brute_force_df["C"], 
         color="black", label="C (Brute Force)")
plt.legend()
plt.title("Brute force solutions")
plt.ylabel("number of molecules")
plt.xlabel("time")

# --- Print out the fraction of successful steps ---
print(f"The fraction of timesteps where reaction 1 is successful is {successful_r1_steps/brute_force_steps}")
print(f"The fraction of timesteps where reaction 2 is successful is {successful_r2_steps/brute_force_steps}")

From the above exercise we can see that we can indeed simulate this system with a "brute force" approach. However, most of our computation is wasted because for most $\Delta t$ there is no change in
the chemical composition of our system. So what can we do better?

**The Efficiency of the Gillespie Algorithm (Stochastic Simulation Algorithm)**
The main problem with the above brute force method is that we waste time checking for a reaction in small $\Delta t$ increments, but we have designed the simaultion in such a way to make the probability small. Thus, most of the time we are copmuting timesteps where nothing of interest occurs. The Gillespie Algorithm solves this by leveraging statistical properties of the reactions:

**1. The Waiting Time is Exponentially Distributed**

Because the reactions are modeled as independent, first-order Poisson processes, the time until the next reaction (of any type) occurs ($\tau$), is itself a random variable that follows an exponential probability distribution. The probability density function (PDF) for this waiting time is:$$P(\tau) = a_0 e^{-a_0 \tau}$$ Gillespie proposed that since we know the probability distribution of the time between events, we no longer have to let that number emerge from our brute force algorithm. Instead, we can directly draw a single value for $\tau$ from the exponential distribution.

As an aside, although Python libraries will often allow you to sample any arbitrary distribution, I think it is useful to understand how to use a uniform distribution to produce any abritrary distribution. We can use a uniformly distributed random number, $\text{rand}_1 \in (0, 1]$, and calculate $\tau$ using the inverse of the cumulative distribution function (CDF):$$\tau = \frac{1}{a_0} \ln\left(\frac{1}{\text{rand}_1}\right)$$Thus, in a single calculation we advance the simulation time by $\tau$, and we are guaranteed that a reaction occurs at that exact time, eliminating all the steps where nothing happens.

**2. Which Reaction Occurs?**

Once $\tau$ is determined, we need to know which of the $N$ reactions (e.g., $A \to B$ or $B \to C$) actually occurs. Just as above, the probability that reaction $i$ is the one that fires is proportional to its individual propensity compared to the total propensity:$$P(\text{Reaction } i) = \frac{a_i}{a_0}$$ Thus, as above, we still select the reaction by drawing a second uniform random number, $\text{rand}_2 \in (0, a_0]$, and finding the reaction index $j$ such that:$$\sum_{i=1}^{j-1} a_i < \text{rand}_2 \le \sum_{i=1}^{j} a_i$$

**3. Summary**

The Gillespie algorithm proposes a two step process: first randomly determine the time between events and then randomly determine which event occurs. By doing so, we ensure that every step taken is productive towards moving the simulation forward and we dramatically reduce our copmutational efforts.

In [ ]:
def gillespie_simulation(A0, B0, C0, k1, k2, T_MAX):
    """
    Gillespie stochastic simulation for A -> B -> C.

    Returns a pandas DataFrame with ['Time', 'A', 'B', 'C'].
    """

    # --- Initialization ---
    time = 0.0
    A_count = A0
    B_count = B0
    C_count = C0

    # Store history
    time_history = [time]
    A_history = [A_count]
    B_history = [B_count]
    C_history = [C_count]

    # --- Simulation Loop ---
    while time < T_MAX:
        # Calculate propensities
        a1 = k1 * A_count
        a2 = k2 * B_count
        a_sum = a1 + a2

        if a_sum == 0: # this only happens when no more reactions can occur
            break

        # --- Throw the dice to see how long until the next reaction ---
        r1 = random.random()
        tau = -np.log(r1) / a_sum # this is the key to Gillespie!!
        time += tau

        # --- Throw the dice to see which reaction occurs --- 
        r2 = random.random() * a_sum
        if r2 < a1 and A_count > 0: # I'm using better logic here ;)
            A_count -= 1
            B_count += 1
        elif B_count > 0: # more "pythonic"
            B_count -= 1
            C_count += 1

        # --- Record state ---
        time_history.append(time)
        A_history.append(A_count)
        B_history.append(B_count)
        C_history.append(C_count)

    # --- Convert to DataFrame ---
    gillespie_df = pd.DataFrame({
        'Time': time_history,
        'A': A_history,
        'B': B_history,
        'C': C_history
    })

    return gillespie_df

In [ ]:
# --- Run the simulation with same parameters as above --- 
gillespie_df = gillespie_simulation(A0, B0, C0, k1, k2, T_MAX)

In [ ]:
# --- Plot the Gillespie results --- 
plt.plot(gillespie_df["Time"], gillespie_df["A"], 
         color="red", label="A (Gillespie)")
plt.plot(gillespie_df["Time"], gillespie_df["B"], 
         color="blue", label="B (Gillespie)")
plt.plot(gillespie_df["Time"], gillespie_df["C"], 
         color="black", label="C (Gillespie)")
plt.ylabel("number of molecules")
plt.xlabel("time")
plt.legend()
plt.title("Gillespie Solutions")
print(f"Total steps taken (Gillespie): {len(gillespie_df)}")

In [ ]:
# --- Plot all three to compare --- 

plt.figure(figsize=(7,5))

# --- Deterministic ---
plt.plot(deterministic_df["Time"], deterministic_df["A"], 
         color="red", label="A (deterministic)", linewidth=5, alpha=0.5)
plt.plot(deterministic_df["Time"], deterministic_df["B"], 
         color="blue", label="B (deterministic)", linewidth=5, alpha=0.5)
plt.plot(deterministic_df["Time"], deterministic_df["C"], 
         color="black", label="C (deterministic)", linewidth=5, alpha=0.5)

# --- Gillespie ---
plt.plot(gillespie_df["Time"], gillespie_df["A"], 
         "r--", label="A (Gillespie)")
plt.plot(gillespie_df["Time"], gillespie_df["B"], 
         "b--", label="B (Gillespie)")
plt.plot(gillespie_df["Time"], gillespie_df["C"], 
         "k--", label="C (Gillespie)")

# --- Brute Force --- 
plt.plot(brute_force_df["Time"], brute_force_df["A"], 
         "r-.", label="A (Brute Force)")
plt.plot(brute_force_df["Time"], brute_force_df["B"], 
         "b-.", label="B (Brute Force)")
plt.plot(brute_force_df["Time"], brute_force_df["C"], 
         "k-.", label="C (Brute Force)")

# --- General --- 
plt.xlabel("Time")
plt.ylabel("Number of molecules")
plt.legend()
plt.title("Comparison: Deterministic vs Gillespie")
plt.tight_layout()
plt.show()

### **Homework for the reader**
1. Prove that the time between events sampled in the Brute Force routine is exponentially distributed.

2. Implement a more complicated reaction scheme where B can go to products C or D with rate constants k2 and k3.

3. Implement a more complicated reaction scheme with reversible reactions.

4. Implement a more complicated reaction scheme with reactions like A + B -> C